In [6]:
import sys
import os

# A very common and generally reliable way if your notebook is in 'notebooks'
# and the package is one level up:
if os.getcwd().endswith('notebooks'): # Check if CWD is the notebooks folder
    project_root = os.path.abspath('..')
else:
    # Fallback or assume project root is CWD if not in 'notebooks'
    # This part might need adjustment based on how/where you launch Jupyter
    project_root = os.getcwd()


# Add the project root to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

In [7]:
import pandas as pd
import os
from libPBL2425NovaNOS.data_access import loader # Assuming your package is 'data_pipeline_nos'
from config import settings

In [ ]:
# --- Test 1: Get Database Engine ---
print("\n[Test 1] Attempting to get database engine...")
try:
    engine = loader.get_db_engine()
    print("Successfully created database engine.")
    # You could try a simple connection test if you want, but fetch_table will do that
    # with engine.connect() as connection:
    #     print("Successfully connected to the database via engine.")
except Exception as e:
    print(f"Error creating database engine: {e}")
    # Stop further tests if engine creation fails
    raise

# --- Test 2: Fetch a Single Table (e.g., masterdatagcs, as it's standalone) ---
# Choose a table that's not too large for a quick test.
# 'masterdatagcs' is one of the tables loaded individually in your original notebook.
print(f"\n[Test 2] Attempting to fetch a single table: '{settings.MASTER_GCS_TABLE_NAME}'...")
try:
    single_table_df = loader.fetch_table_from_db(settings.MASTER_GCS_TABLE_NAME, engine)
    print(f"Successfully fetched '{settings.MASTER_GCS_TABLE_NAME}'.")
    print(f"Shape of '{settings.MASTER_GCS_TABLE_NAME}': {single_table_df.shape}")
    if not single_table_df.empty:
        print(f"Head of '{settings.MASTER_GCS_TABLE_NAME}':")
        print(single_table_df.head())
    else:
        print(f"Warning: Fetched table '{settings.MASTER_GCS_TABLE_NAME}' is empty.")
    assert not single_table_df.empty, f"{settings.MASTER_GCS_TABLE_NAME} should not be empty" # Basic check
except Exception as e:
    print(f"Error fetching table '{settings.MASTER_GCS_TABLE_NAME}': {e}")
    raise

# --- Test 3: Load and Concatenate All Raw Data ---
print("\n[Test 3] Attempting to load and concatenate all raw data...")
try:
    raw_dataframes = loader.load_and_concatenate_raw_data()
    print("Successfully executed load_and_concatenate_raw_data().")

    # Verify the output structure
    expected_keys = ["clients_df", "calls_df", "masterdatagcs_df"]
    assert all(key in raw_dataframes for key in expected_keys), \
        f"Expected keys {expected_keys} not all found in returned dictionary."
    print(f"Returned dictionary contains expected keys: {list(raw_dataframes.keys())}")

    # Extract DataFrames
    clients_df_raw = raw_dataframes['clients_df']
    calls_df_raw = raw_dataframes['calls_df']
    masterdatagcs_df_raw = raw_dataframes['masterdatagcs_df']

    # Print shapes and head for each major DataFrame
    print("\n--- Raw Clients DataFrame ---")
    print(f"Shape: {clients_df_raw.shape}")
    if not clients_df_raw.empty:
        print("Head:")
        print(clients_df_raw.head())
        # Compare with expected row count if known from original notebook
        # assert clients_df_raw.shape[0] > 0, "Clients DataFrame should have rows."
    else:
        print("Warning: Raw Clients DataFrame is empty.")


    print("\n--- Raw Calls DataFrame ---")
    print(f"Shape: {calls_df_raw.shape}")
    if not calls_df_raw.empty:
        print("Head:")
        print(calls_df_raw.head())
        # Compare with expected row count if known from original notebook
        # assert calls_df_raw.shape[0] > 0, "Calls DataFrame should have rows."
    else:
        print("Warning: Raw Calls DataFrame is empty.")

    print("\n--- Raw MasterdataGCS DataFrame ---")
    print(f"Shape: {masterdatagcs_df_raw.shape}")
    if not masterdatagcs_df_raw.empty:
        print("Head:")
        print(masterdatagcs_df_raw.head())
        # Compare with expected row count if known from original notebook
        # assert masterdatagcs_df_raw.shape[0] > 0, "MasterdataGCS DataFrame should have rows."
    else:
        print("Warning: Raw MasterdataGCS DataFrame is empty.")

    # Basic assertions for non-emptiness
    assert not clients_df_raw.empty, "clients_df_raw should not be empty"
    assert not calls_df_raw.empty, "calls_df_raw should not be empty"
    assert not masterdatagcs_df_raw.empty, "masterdatagcs_df_raw should not be empty"

    print("\nAll basic loader tests passed!")

except Exception as e:
    print(f"An error occurred during load_and_concatenate_raw_data testing: {e}")
    raise

# --- Optional: Save to Interim for Next Stage (if desired) ---
# This is useful if the loading process is slow and you want to
# start the cleaning notebook with already loaded data.
# Make sure the INTERIM_DATA_DIR is created by settings.py

# print(f"\nOptional: Saving raw dataframes to {settings.INTERIM_DATA_DIR}...")
# try:
#     clients_df_raw.to_csv(settings.RAW_CLIENTS_FILE, index=False)
#     calls_df_raw.to_csv(settings.RAW_CALLS_FILE, index=False)
#     masterdatagcs_df_raw.to_csv(settings.RAW_GCS_FILE, index=False)
#     print("Raw dataframes saved to interim directory.")
# except Exception as e:
#     print(f"Error saving interim files: {e}")

Fetching tables from database...
Fetching masterdataclients_jan_may...


OperationalError: (psycopg2.OperationalError) connection to server at "172.20.20.4", port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)